# log_003 · The EEG pipeline, and why preprocessing exists

*Raw → Evoked · tutorial series, part 1 of 5*

---

**The one idea to hold onto before anything else:**

> EEG is a tiny brain signal buried under mountains of non-brain noise.
> Preprocessing is not "tidying up" — it's the fight to pull microvolts of
> real neural activity out from under millivolts of junk sitting on top of it.

Every step below exists to remove **one specific kind of contamination** without
damaging the real signal underneath. Once you hold that idea, each step stops
being a mystery and becomes an obvious answer to: *what garbage am I removing, and why?*

In this notebook we run the whole pipeline once, end to end, on real data — the
classic **eyes-open vs. eyes-closed** experiment. When you close your eyes, neurons
in the occipital lobe (the brain's visual area, at the back of your head) synchronise
at around 10 Hz, producing **alpha waves**. Open your eyes and they collapse. Our goal:
make that alpha spike appear in a plot.

The deep-dives (references, filters, artifacts, epoch rejection) each get their own
tutorial. This one is the map.

## 0 · Setup

We only need a small, free stack: `mne` for everything EEG, plus `numpy` and
`matplotlib`. If you're running this fresh, uncomment the install line.

`mne` can download well-known public datasets for us, so there are **no files to
hunt down** — the first run fetches the data automatically.

In [ ]:
# !pip install mne numpy matplotlib

import mne
import numpy as np
import matplotlib.pyplot as plt

# Keep MNE's console output quiet-ish so the notebook stays readable.
mne.set_log_level("WARNING")

print("MNE version:", mne.__version__)

## 1 · Load the data — and set the montage

**What we do:** download two baseline runs for one subject from the PhysioNet
*EEG Motor Movement/Imagery* dataset. Run 1 is **eyes open**, run 2 is **eyes closed**.
Same person, same electrodes, only the eyes differ — a clean before/after.

**Why the montage matters:** an EEG number is meaningless until you know *where*
on the head each electrode sat. The montage maps channel names (like `O1`, `Cz`)
to physical scalp positions using the standard **10-20 system**. Without it you
can't make a head-map or interpret anything spatially. Think of the channel names
as a coordinate system, not just labels.

In [ ]:
# Subject 1, runs 1 (eyes open) and 2 (eyes closed)
raw_fnames = mne.datasets.eegbci.load_data(subjects=1, runs=[1, 2])

raw_open = mne.io.read_raw_edf(raw_fnames[0], preload=True)
raw_closed = mne.io.read_raw_edf(raw_fnames[1], preload=True)

# This dataset's channel names have trailing dots (e.g. "Fpz.") — clean them,
# then attach the standard 10-20 positions so MNE knows the geometry.
for raw in (raw_open, raw_closed):
    mne.datasets.eegbci.standardize(raw)          # fixes channel names
    raw.set_montage("standard_1005")              # gives every channel a 3D location

print("Eyes-open :", raw_open)
print("Eyes-closed:", raw_closed)
print("\nChannels:", raw_open.ch_names[:8], "...")
print("Sampling rate:", raw_open.info["sfreq"], "Hz")

Let's actually *look* at the raw signal before touching it. This is the
"messy start" — drift, noise, and somewhere in there, brain activity.

In [ ]:
# Plot ~5 seconds of a few channels, eyes closed.
raw_closed.plot(duration=5, n_channels=8, scalings="auto",
                title="Raw EEG (eyes closed) — before any cleaning", show=True)

## 2 · Re-reference

**What we do:** re-reference to the **average** of all channels.

**Why:** every EEG voltage is a *difference* between the recording electrode and a
reference point — there's no such thing as absolute brain voltage, only voltage
*relative to somewhere*. If that "somewhere" is noisy, the noise leaks into every
channel. The average reference uses the mean of all electrodes as a cleaner, more
neutral baseline.

*(There are other choices — linked mastoids, REST, single-electrode — and which one
you pick depends mostly on how many electrodes you have. That's the whole of
Tutorial 2. For a 64-channel set like this, the average reference is a sensible
default.)*

In [ ]:
for raw in (raw_open, raw_closed):
    raw.set_eeg_reference("average", projection=False)

print("Re-referenced both runs to the average of all channels.")

## 3 · Filter

**What we do:** apply a **band-pass** filter keeping roughly **1–40 Hz**.

**Why:** three problems, handled at once —
- **below ~1 Hz:** slow drift from sweat and electrodes settling → removed by the
  high-pass edge.
- **above ~40 Hz:** fast muscle noise → removed by the low-pass edge.
- **our target, alpha (8–12 Hz),** sits comfortably inside the band we keep.

**The danger to respect:** filters have side effects. Too aggressive a high-pass can
distort real signal shapes. The craft isn't "apply filter" — it's "remove the noise
band *without* eating the signal band next to it." (Filter types and design tradeoffs
are Tutorial 3.)

We're skipping a dedicated **notch** filter here for simplicity; the 40 Hz low-pass
already suppresses 50/60 Hz line noise. In a country with strong mains hum you'd add
`raw.notch_filter(50)` (or `60` in the Americas).

In [ ]:
for raw in (raw_open, raw_closed):
    raw.filter(l_freq=1.0, h_freq=40.0)

print("Band-pass filtered both runs to 1-40 Hz.")

# Look at the same eyes-closed data again — notice how much flatter and cleaner
# the baseline is now compared to the raw plot above.
raw_closed.plot(duration=5, n_channels=8, scalings="auto",
                title="After filtering (eyes closed) — drift and fast noise gone",
                show=True)

## 4 · A note on ICA (and why we skip it *this time*)

The full pipeline includes **ICA** to remove eye-blinks and heartbeat — it un-mixes
the recording into independent sources so you can delete just the bad ones. It's the
centrepiece of Tutorial 4.

But here's an honest teaching moment: **the alpha effect is so strong it survives
minimal cleaning.** For this first pass we'll skip ICA and still see a clear result.
Later, re-running *with* ICA and showing the plot get even cleaner makes a great
follow-up — the contrast teaches more than either version alone.

So: note that ICA belongs *here* in the order (after filtering, before epoching),
and move on.

## 5 · The payoff — compute the power spectrum

We don't strictly need to cut this resting data into event-locked epochs (there are
no events — the subject just sat with eyes open or closed). Instead we go straight to
the question: **how much power is there at each frequency, and does it differ between
conditions?**

**What we do:** compute the **power spectral density (PSD)** — power at each frequency —
using Welch's method, built into MNE. Then we focus on the **occipital channels**
(`O1`, `O2`, `Oz`), because that's where alpha lives.

**Why occipital:** alpha is generated in the visual cortex at the back of the head.
Looking at frontal channels would dilute the effect. Knowing *where* to look is half
the analysis — and it's why we bothered with the montage in step 1.

In [ ]:
# The occipital channels — the back of the head, home of alpha.
occipital = ["O1", "O2", "Oz"]

# Compute PSD for each condition, restricted to a sensible frequency window.
psd_open = raw_open.compute_psd(method="welch", fmin=1, fmax=40, picks=occipital)
psd_closed = raw_closed.compute_psd(method="welch", fmin=1, fmax=40, picks=occipital)

# Average across the three occipital channels to get one clean curve each.
freqs = psd_open.freqs
power_open = psd_open.get_data().mean(axis=0)
power_closed = psd_closed.get_data().mean(axis=0)

print("Computed PSD over occipital channels for both conditions.")

Now the moment of truth. If the physiology is real and our pipeline is sound,
the **eyes-closed** curve should spike sharply around **10 Hz** while the eyes-open
curve stays flat there.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Convert to dB for readability (standard for PSD plots).
ax.plot(freqs, 10 * np.log10(power_open), label="Eyes open",
        color="#8a8178", linewidth=2)
ax.plot(freqs, 10 * np.log10(power_closed), label="Eyes closed",
        color="#b8482e", linewidth=2)

# Shade the alpha band so the reader's eye goes straight to it.
ax.axvspan(8, 12, color="#b8482e", alpha=0.08, label="Alpha band (8-12 Hz)")

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power (dB)")
ax.set_title("Occipital power: eyes open vs. eyes closed")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

**What you should see:** a clear bump in the red (eyes-closed) curve right inside
the shaded 8-12 Hz band, with the grey (eyes-open) curve flat there. That bump is
**your alpha rhythm** — the synchronised firing of your visual cortex when it has
nothing to look at.

That is the entire point: a real, physiological brain signal, pulled cleanly out of
the noise by a pipeline you now understand step by step.

## 6 · The hero image — where on the head is the alpha?

A spectrum proves alpha is *there*. A **topographic map** proves it's in the *right
place* — the back of the head. This is the image worth putting at the top of your
write-up.

**What we do:** compute alpha-band power across *all* channels for the eyes-closed
run and paint it onto a head seen from above. We expect a hotspot at the occipital
pole.

In [ ]:
# PSD across ALL channels, eyes closed, limited to the alpha band.
psd_all = raw_closed.compute_psd(method="welch", fmin=8, fmax=12)
alpha_power = psd_all.get_data().mean(axis=1)  # average power within 8-12 Hz per channel

fig, ax = plt.subplots(figsize=(5, 5))
mne.viz.plot_topomap(alpha_power, raw_closed.info, axes=ax, show=False,
                     cmap="Reds", contours=4)
ax.set_title("Alpha power (8-12 Hz), eyes closed")
plt.tight_layout()
plt.show()

**What you should see:** the red hotspot concentrated at the **back of the head** —
exactly where the occipital lobe sits. If the glow were at the front, you'd suspect an
artifact (frontal usually means eye or muscle). The fact that it lands at the back is
the physiology confirming your pipeline worked.

## 7 · What we learned

Walking the pipeline once, the shape of it:

1. **Load + montage** — give every electrode a position, or nothing spatial works.
2. **Re-reference** — voltage is always relative; choose a neutral baseline.
3. **Filter** — remove drift, muscle, and line noise *without* eating the signal.
4. **(ICA)** — un-mix and remove blinks/heartbeat; skippable when the effect is strong.
5. **Analyse** — go where the signal lives (occipital, for alpha) and let averaging
   or spectral estimation reveal what's invisible in the raw trace.

**Two ideas to carry into every future analysis:**

- **Order isn't arbitrary.** You filter before ICA because ICA works better on cleaner
  input; you'd reject bad epochs before averaging so they don't poison the mean. Each
  step assumes the last one ran.
- **Every step is a trade-off, never free.** Filtering removes noise but can distort
  signal. ICA removes artifacts but can delete real brain data if you drop the wrong
  component. Good preprocessing isn't running the *most* steps — it's knowing the
  *minimum* needed and what each one costs.

**Next in the series →** *Every EEG reference, and when to use each* — we take this exact
data, apply average / linked-mastoid / REST references, and put the head-maps side by
side to *see* how the choice changes the result.

---
*Open source, built in the open. Fork it, run it, tell me where I'm wrong.*